# LBSN2Vec++ social track — stages 0–4 + RotH — LBSN_NYC

Runs the full new-dataset pipeline and trains the hyperbolic embeddings. The centerpiece is
**stage 3 (RotH + depth regulariser)**; stages 0–2 rebuild its inputs from scratch unless you
attach them prebuilt.

| Stage | Script | Produces | Time (4-core CPU) |
|---|---|---|---|
| 0 | `prepare_lbsn_csvs.py` | house-format CSVs from the raw 2.68 GB dump | ~10 min (skipped if attached) |
| 1 | `build_groups.py` | co-location groups, tie graph, group examples | ~10–15 min |
| 2 | `build_kg_lbsn.py` | POI + group + **FRIEND_OF** knowledge graph | ~3 min |
| 3 | `train_roth.py` | `poi_hyperbolic_embs_LBSN_NYC.npy` (drop-in `EMB_FILE`) | **~2.5–4 h** @ 120 epochs |
| 4 | `build_poi_poi_triples.py` | `poi_poi_triples_LBSN_NYC.pt` for §6b alignment | ~2 min |

**CPU only — set the accelerator to `None`.** No GPU, no HF token.

## What to attach as Kaggle input

**1. Code (required)** — a private Kaggle Dataset containing `src/*.py` from the repo **plus
`data/lbsn/fsq_category_paths_2014.json`** (the 2014 Foursquare taxonomy; without it every
category is flat and the depth regulariser has nothing to order).

**2. Data — one of, in order of preference:**
- the prepared CSVs (`train/val/test_LBSN_NYC.csv`, `poi_metadata_LBSN_NYC.csv`,
  `friendship_old_LBSN_NYC.csv`, `friendship_new_only_LBSN_NYC.csv` from the repo's
  `data/lbsn/`) → stage 0 is skipped entirely, **no Internet needed**;
- the raw `lsbn2vec_global.zip` as a dataset → stage 0 runs from it, no Internet needed;
- nothing + **Internet ON** → stage 0 downloads the zip via gdown (~1 min) and runs.

**You paste no paths.** Cell 0 finds everything by content.

> Every code cell reloads its own state from `_paths.json`, so a kernel restart or
> out-of-order execution will not break it. Cell 0 still has to run once per session.


## 0 · Locate inputs  *(run this once per session)*

In [ ]:
import os, sys, glob, json, shutil, subprocess, zipfile, urllib.request

DATASET = "LBSN_NYC"
GDRIVE_ZIP_ID = "1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8"     # the raw dataset_WWW2019 dump
TAXONOMY_GIST = ("https://gist.githubusercontent.com/tndoan/"
                 "24ea389efbe137d9a514/raw")            # fallback only

# everything stored in _paths.json must be ABSOLUTE: run_stream launches scripts with
# cwd=RUN, so a relative path would resolve against the wrong directory when run locally
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
os.makedirs(WORK, exist_ok=True)

def _find(pred):
    for root in ("/kaggle/input", WORK, "."):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if pred(set(files)):
                return os.path.abspath(dirpath)
    return None

# -- code: all of src/*.py -----------------------------------------------------------
CODE_DIR = _find(lambda fs: {"build_kg_lbsn.py", "prepare_lbsn_csvs.py"} <= fs)
assert CODE_DIR, "no attached dataset contains src/*.py (build_kg_lbsn.py etc.) -- upload it"
RUN = f"{WORK}/code_run"
os.makedirs(RUN, exist_ok=True)
# copyfile + chmod, NOT shutil.copy: copy() preserves the read-only bits from /kaggle/input.
# The same-file guard covers re-running this cell after _find resolves CODE_DIR to RUN itself.
for src in glob.glob(f"{CODE_DIR}/*.py"):
    dst = os.path.join(RUN, os.path.basename(src))
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copyfile(src, dst)
        os.chmod(dst, 0o644)
sys.path.insert(0, RUN)
need = {"prepare_lbsn_csvs.py", "build_groups.py", "affinity.py", "build_kg.py",
        "build_kg_lbsn.py", "train_roth.py", "build_poi_poi_triples.py"}
have = {os.path.basename(p) for p in glob.glob(f"{RUN}/*.py")}
assert need <= have, f"code dataset is missing {sorted(need - have)}"
print("CODE_DIR :", CODE_DIR)

# -- taxonomy: find the committed JSON, else rebuild it from the gist (Internet ON) ----
TAXONOMY = None
tax_dir = _find(lambda fs: "fsq_category_paths_2014.json" in fs)
if tax_dir:
    TAXONOMY = os.path.join(tax_dir, "fsq_category_paths_2014.json")
else:
    try:
        txt = urllib.request.urlopen(TAXONOMY_GIST, timeout=60).read().decode()
        by_name, stack = {}, []
        for line in txt.splitlines():
            if not line.strip() or line.strip().startswith("Suggested Countries"):
                continue
            indent = (len(line) - len(line.lstrip(" "))) // 4
            stack = stack[:indent] + [line.strip()]
            by_name.setdefault(stack[-1], []).append(">".join(stack))
        TAXONOMY = f"{WORK}/fsq_category_paths_2014.json"
        json.dump({n: sorted(set(p)) for n, p in by_name.items()}, open(TAXONOMY, "w"))
        print("taxonomy rebuilt from the gist:", len(by_name), "category names")
    except Exception as e:
        raise SystemExit(
            "fsq_category_paths_2014.json not attached and the gist fetch failed "
            f"({e}) -- without the taxonomy the KG is flat and the depth regulariser "
            "is pointless. Attach the JSON (it is committed at data/lbsn/).") from e
print("TAXONOMY :", TAXONOMY)

# -- data: prepared CSVs > raw zip > download-later ------------------------------------
prepared = {f"train_{DATASET}.csv", f"poi_metadata_{DATASET}.csv",
            f"friendship_old_{DATASET}.csv"}
LBSN_DIR = _find(lambda fs: prepared <= fs)
ZIP_PATH = None
if LBSN_DIR:
    print("LBSN_DIR :", LBSN_DIR, " (prepared CSVs attached -- stage 0 will be skipped)")
else:
    LBSN_DIR = f"{WORK}/lbsn"                       # stage 0 will fill it
    for root in ("/kaggle/input", WORK):
        if ZIP_PATH or not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if ZIP_PATH:
                break
            for f in files:
                if not f.endswith(".zip"):
                    continue
                p = os.path.join(dirpath, f)
                try:
                    with zipfile.ZipFile(p) as z:
                        if "dataset_WWW2019/raw_POIs.txt" in z.namelist():
                            ZIP_PATH = p
                            break
                except Exception:
                    pass
    print("LBSN_DIR :", LBSN_DIR, " (to be built by stage 0)")
    print("ZIP_PATH :", ZIP_PATH or "none attached -- stage 0 will gdown it (Internet ON)")

GROUPS_DIR, KG_DIR = f"{WORK}/groups", f"{WORK}/kg"
json.dump(dict(RUN=RUN, LBSN_DIR=LBSN_DIR, KG_DIR=KG_DIR, GROUPS_DIR=GROUPS_DIR,
               WORK=WORK, DATASET=DATASET, TAXONOMY=TAXONOMY, ZIP_PATH=ZIP_PATH,
               GDRIVE_ZIP_ID=GDRIVE_ZIP_ID),
          open(f"{WORK}/_paths.json", "w"), indent=1)
print("\npaths written to", f"{WORK}/_paths.json")


## 1 · Self-checks

Every script ships a synthetic fixture — seconds each, and they fail loudly on a broken
environment instead of 2 hours into training. Expect 10 + 11 + 12 + 15 + 4 + 9 =
**61 PASS, 0 FAIL**.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

total_pass = total_fail = 0
for script in ("prepare_lbsn_csvs.py", "build_groups.py", "build_kg.py",
               "build_kg_lbsn.py", "train_roth.py", "build_poi_poi_triples.py"):
    r = subprocess.run([sys.executable, f"{RUN}/{script}", "--self-check"],
                       capture_output=True, text=True, cwd=RUN)
    lines = [l for l in r.stdout.splitlines() if l.strip().startswith(("PASS", "FAIL"))]
    n_p = sum(l.strip().startswith("PASS") for l in lines)
    n_f = sum(l.strip().startswith("FAIL") for l in lines)
    total_pass += n_p; total_fail += n_f
    print(f"{script:<26} exit={r.returncode}  PASS={n_p}  FAIL={n_f}")
    for l in lines:
        if l.strip().startswith("FAIL"):
            print("   ", l.strip())
    if r.returncode != 0:
        print(r.stdout[-2000:]); print("STDERR:", r.stderr[-1000:])
print(f"\ntotal: {total_pass} PASS, {total_fail} FAIL")
assert total_fail == 0 and total_pass >= 55, "self-checks failed -- do not proceed"


## 2 · Stage 0 — raw dump → house-format CSVs

Skipped automatically when the prepared CSVs are attached. Otherwise streams through the zip
(POI bbox filter → check-in filter → venues ≥ 10 visits → users ≥ 30) and writes the splits.

Expected (identical to the `preprocessing-global-fsq` notebook, which this reproduces):

```
POIs in bbox 102,687 → check-ins 288,188 → after filters 159,304 / 1,665 users / 6,103 POIs
friendship_old 1,506 edges (KG input) · friendship_new_only 1,138 pairs (EVAL ONLY, never in the KG)
splits  train 112,242 / val 15,853 / test 31,209
taxonomy depths over POIs  {1: 236, 2: 4,231, 3: 1,608, 4: 28}
```

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

import pandas as pd

prepared = [f"train_{DATASET}.csv", f"val_{DATASET}.csv", f"test_{DATASET}.csv",
            f"poi_metadata_{DATASET}.csv", f"friendship_old_{DATASET}.csv",
            f"friendship_new_only_{DATASET}.csv"]
if all(os.path.exists(os.path.join(LBSN_DIR, f)) for f in prepared):
    print("prepared CSVs found in", LBSN_DIR, "-- skipping stage 0")
else:
    zp = ZIP_PATH                                            # noqa: F821  (from _paths.json)
    if not zp:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        zp = f"{WORK}/lsbn2vec_global.zip"
        subprocess.run([sys.executable, "-m", "gdown", "--id", GDRIVE_ZIP_ID,   # noqa: F821
                        "-O", zp], check=True)
    os.makedirs(LBSN_DIR, exist_ok=True)
    run_stream("prepare_lbsn_csvs.py", "--zip", zp, "--out-dir", LBSN_DIR,
               "--dataset", DATASET, "--taxonomy", TAXONOMY)

# verify against the reference counts, whichever path was taken
n = {s: len(pd.read_csv(os.path.join(LBSN_DIR, f"{s}_{DATASET}.csv")))
     for s in ("train", "val", "test")}
meta = pd.read_csv(os.path.join(LBSN_DIR, f"poi_metadata_{DATASET}.csv"))
fo = pd.read_csv(os.path.join(LBSN_DIR, f"friendship_old_{DATASET}.csv"))
expect = dict(train=112242, val=15853, test=31209, pois=6103, friends_old=1506)
got = dict(**n, pois=len(meta), friends_old=len(fo))
for k, v in expect.items():
    print(f"  {k:<12} {got[k]:>8,}  (expected {v:,})  {'OK' if got[k] == v else 'DIFFERS'}")


## 3 · Stage 1 — group construction  *(unchanged script, new dataset)*

Real timestamps mean the co-location mining is exactly what `build_groups.py` already does.
`--no-resplit` honours stage 0's per-user 70/10/20.

Expected:

```
real ephemeral groups 3,835 (max span ≤ 60 min) · co_attended pairs 5,997
real group transitions 64  (a curiosity set, still far too few to train on)
group examples  train 25,847 / val 5,782 / test 12,204
all causality / clique / recurrence asserts must pass
```

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

run_stream("build_groups.py", "--data-dir", LBSN_DIR, "--out-dir", GROUPS_DIR,
           "--dataset", DATASET, "--no-resplit")
for f in sorted(os.listdir(GROUPS_DIR)):
    print(f"  {os.path.getsize(f'{GROUPS_DIR}/{f}'):>12,}  {f}")


## 4 · Stage 2 — the knowledge graph (POI + group + FRIEND_OF)

`build_kg.py`'s layers plus the one relation this dataset genuinely adds: **FRIEND_OF**, real
declared friendships from the BEFORE-period snapshot only. The 1,138 `friendship_new`-only
pairs are the friendship-prediction eval set and are **asserted absent** from the triples.

Expected: **12,709 entities · 244,080 triples · 12 relations**, and per relation:

```
IS_NEAR_TO 75,236   FOLLOWED_BY 72,753   VISITED 48,848   CO_ATTENDED 11,994
MEMBER_OF   9,063   LOCATED_IN   6,879   HAS_CATEGORY 6,103   OCCURRED_AT 3,835
GROUP_PREFERS 3,835   FRIEND_OF 3,012   PREFERS_CATEGORY 2,211   SUBCATEGORY_OF 311
PREFERS_CATEGORY depth spread d1:1683 d2:416 d3:111 d4:1   <- users get distinct radii
```

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

run_stream("build_kg_lbsn.py", "--csv-dir", LBSN_DIR, "--groups-dir", GROUPS_DIR,
           "--out-dir", KG_DIR, "--dataset", DATASET, "--taxonomy", TAXONOMY)

man = json.load(open(f"{KG_DIR}/kg_manifest.json"))
expect = dict(n_entities=12709, n_triples=244080, n_relations=12)
print()
for k, v in expect.items():
    got = man[k]
    print(f"  {k:<12} {got:>8,}  (expected {v:,})  {'OK' if got == v else 'DIFFERS'}")
assert "FRIEND_OF" in man["relation_counts"], "FRIEND_OF missing from the KG"
assert man["n_friendship_new_only"] > 0, "leakage guard did not run"
print(f"  leakage guard verified {man['n_friendship_new_only']:,} "
      "friendship_new-only pairs absent")


## 5 · Stage 3 — RotH with the depth regulariser  *(the long cell)*

**~2.5–4 h at 120 epochs on 4 CPU cores** (the graph is 244k triples, 1.4× the TSMC one).
`EPOCHS = 20` is a ~30-min sanity pass. Progress prints every 10 epochs.

What to watch:
- **`D1_rho` should climb.** A 2-epoch smoke run on this exact KG already reached **+0.69**;
  the TSMC run ended at +0.85. If it is still ABSENT past epoch 40, stop the cell and raise
  `--depth-weight` rather than waiting.
- **A high ρ with non-monotonic per-depth radii is a false pass** — read the radii table at
  the end, not just ρ (cell 6 asserts this).
- Overall link-prediction MRR will look *worse* than a no-regulariser run — that is the known
  trade-off: hierarchical relations gain, flat ones (IS_NEAR_TO, FOLLOWED_BY, MEMBER_OF) pay.
  Judge stage 3 on D1 + the per-relation table, never the pooled MRR.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

EPOCHS = 120        # 20 for a ~30-min sanity pass; 120 is the real run

run_stream("train_roth.py", "--kg-dir", KG_DIR, "--data-dir", LBSN_DIR, "--out-dir", KG_DIR,
           "--dataset", DATASET, "--epochs", EPOCHS, "--log-every", 10, "--max-eval", 4000,
           "--depth-weight", 5.0, "--depth-margin", 0.3, "--root-pull", 0.01)


## 6 · D1 — did the ball get a radial hierarchy?

The pass bar, in order of authority: **monotone radii by depth** (d1 < d2 < d3 < d4), then
ρ ≥ +0.8 STRONG. The 2-epoch smoke gave `0.666 → 0.681 → 0.870 → 1.209` at ρ = +0.69, so the
full run should clear both comfortably. Depth 4 is only 28 POIs — a wobble there with the
rest monotone is acceptable; a wobble at d1–d3 is not.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

import numpy as np, pandas as pd
from train_roth import d1_radial_hierarchy

meta = pd.read_csv(f"{LBSN_DIR}/poi_metadata_{DATASET}.csv")
col = "category_path" if "category_path" in meta.columns else "category"
depths = np.array([len([x for x in str(s).split(">") if x.strip()])
                   for s in meta[col].fillna("")], dtype=float)

d = d1_radial_hierarchy(np.load(f"{KG_DIR}/poi_hyperbolic_embs_{DATASET}.npy"), depths)
print(f"D1 rho={d['spearman']:+.4f}  {d['verdict']}  "
      f"norms [{d['norm_min']:.3f}, {d['norm_max']:.3f}]")
for k, v in d["by_depth"].items():
    print(f"   depth {k}: n={v['n']:>5}  mean radius = {v['mean_radius']:.4f}")

radii = [v["mean_radius"] for v in d["by_depth"].values()]
mono_core = all(radii[i] < radii[i + 1] for i in range(min(len(radii) - 1, 2)))
print(f"\nmonotone over d1-d3: {mono_core}   full: "
      f"{all(radii[i] < radii[i+1] for i in range(len(radii)-1))}")
assert mono_core and d["verdict"] == "STRONG", (
    "D1 failed -- do NOT ship these embeddings; raise --depth-weight and rerun")
print("PASS -- poi_hyperbolic_embs_%s.npy is good to ship" % DATASET)


## 7 · Optional — the `--depth-weight 0` control

The row that turns stage 3 from an assertion into an argument (on TSMC the control came out
ρ = −0.11, ABSENT). **Doubles the runtime**; flip `RUN_CONTROL = True` only if the session has
the hours to spare. It trains into a separate directory and never touches the real outputs.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

RUN_CONTROL = False

if RUN_CONTROL:
    CTRL = f"{KG_DIR}_control"
    run_stream("train_roth.py", "--kg-dir", KG_DIR, "--data-dir", LBSN_DIR,
               "--out-dir", CTRL, "--dataset", DATASET, "--epochs", 120,
               "--log-every", 10, "--max-eval", 4000,
               "--depth-weight", 0, "--depth-margin", 0.3, "--root-pull", 0)
    import numpy as np, pandas as pd
    from train_roth import d1_radial_hierarchy
    meta = pd.read_csv(f"{LBSN_DIR}/poi_metadata_{DATASET}.csv")
    col = "category_path" if "category_path" in meta.columns else "category"
    depths = np.array([len([x for x in str(s).split(">") if x.strip()])
                       for s in meta[col].fillna("")], dtype=float)
    for name, path in (("regularised", KG_DIR), ("control (dw=0)", CTRL)):
        d = d1_radial_hierarchy(
            np.load(f"{path}/poi_hyperbolic_embs_{DATASET}.npy"), depths)
        print(f"{name:<16} rho={d['spearman']:+.4f}  {d['verdict']}")
else:
    print("skipped -- set RUN_CONTROL = True to run it")


## 8 · Stage 4 — POI-POI triples for the alignment loss

Feeds the KG-triple-preservation term in `stage6b_run2_server.ipynb` §6b. `--derive taxonomy`
matters: both native POI-POI relations are flat, so without the derived taxonomy-sibling
relations the TransE term has no hierarchy to preserve.

Expected: **132,394 triples over 5 relations**
(`FOLLOWED_BY` 40,000 · `IS_NEAR_TO` 40,000 · `SAME_TAXONOMY_L2` 40,000 ·
`SAME_TAXONOMY_L3` 12,170 · `SAME_TAXONOMY_L4` 224).

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

run_stream("build_poi_poi_triples.py", "--kg-dir", KG_DIR,
           "--meta", f"{LBSN_DIR}/poi_metadata_{DATASET}.csv",
           "--out-dir", KG_DIR, "--dataset", DATASET,
           "--derive", "taxonomy", "--max-per-relation", 40000)

v = json.load(open(f"{KG_DIR}/poi_relation_vocab_{DATASET}.json"))
print("\nrelations in the alignment triple set:")
for k, n in v["counts"].items():
    print(f"   {k:<22} {n:>8,}")


## 9 · Outputs — what to save and what goes where

**Save a Version** so the output can be attached to the next notebook as a dataset, and copy
these back into the repo:

| File | Goes to | Used as |
|---|---|---|
| `kg/poi_hyperbolic_embs_LBSN_NYC.npy` | `data/kg_lbsn/` | `EMB_FILE` for the fine-tune |
| `kg/roth_best.pt`, `kg/roth_results.json` | `data/kg_lbsn/` | checkpoint + the D1/MRR record |
| `kg/poi_poi_triples_LBSN_NYC.pt` + `kg/poi_relation_vocab_LBSN_NYC.json` | `data/kg_lbsn/` | §6b `ALIGN_TRIPLES_FILE` / `ALIGN_RELVOCAB_FILE` |
| `groups/group_examples_{train,val,test}.jsonl` | `data/lbsn/groups/` | the group task data |
| `lbsn/*.csv` (if stage 0 ran here) | `data/lbsn/` | the canonical splits + friendship files |

The embeddings file is `(6103, 64)` float32, rows in `poi_idx` order. `roth_results.json`
carries the full config, the per-relation link-prediction table, and the D1 record — keep it
next to the embeddings; it is the provenance for every number the paper will cite.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, LBSN_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET, TAXONOMY   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 3-hour training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

for d in (LBSN_DIR, GROUPS_DIR, KG_DIR):
    if not os.path.isdir(d):
        continue
    print(d)
    for f in sorted(os.listdir(d)):
        p = os.path.join(d, f)
        if os.path.isfile(p):
            print(f"   {os.path.getsize(p):>12,}  {f}")
    print()

import numpy as np
emb = np.load(f"{KG_DIR}/poi_hyperbolic_embs_{DATASET}.npy")
res = json.load(open(f"{KG_DIR}/roth_results.json"))
print(f"embeddings {emb.shape} {emb.dtype}   "
      f"D1 rho={res['d1']['spearman']:+.4f} ({res['d1']['verdict']})   "
      f"MRR={res['link_prediction']['mrr']:.4f}")
